# analysis model difference explanations v1

目标：验证 `src/analysis.build_model_difference_explanations` 能把 regime/window 的模型 winner、loser、基准差异和 review 建议整理成稳定中文解释表。

边界：本实验读取 QuantBench smoke 回测 sidecar，只验证框架解释能力；不把 smoke 结果当作真实生产模型优劣。

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists():
        REPO_ROOT = candidate
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import analyze_model_effects_by_regime_and_window, build_model_difference_explanations

CARD = REPO_ROOT / 'research/analysis_model_difference_explanations_v1'
SOURCE_CARD = REPO_ROOT / 'research/quantbench_model_matrix_smoke_v1'
CARD.mkdir(parents=True, exist_ok=True)

daily = pd.read_csv(SOURCE_CARD / 'daily_backtest.csv', parse_dates=['date'])
comparison = pd.read_csv(SOURCE_CARD / 'model_comparison.csv')
baseline_model = str(comparison.iloc[0]['model'])
min_date = daily['date'].min()
max_date = daily['date'].max()
mid_date = min_date + (max_date - min_date) / 2
windows = {
    'early_test': (str(min_date.date()), str(mid_date.date())),
    'late_test': (str((mid_date + pd.Timedelta(days=1)).date()), str(max_date.date())),
    'full_test': (str(min_date.date()), str(max_date.date())),
}
baseline_model, windows

('logistic_clf',
 {'early_test': ('2026-05-01', '2026-05-20'),
  'late_test': ('2026-05-21', '2026-06-08'),
  'full_test': ('2026-05-01', '2026-06-08')})

In [2]:
analysis = analyze_model_effects_by_regime_and_window(
    daily,
    windows=windows,
    baseline_model=baseline_model,
)
explanations = build_model_difference_explanations(analysis)
explanations.to_csv(CARD / 'model_difference_explanations.csv', index=False)

signal_counts = explanations['explanation_signal'].value_counts().sort_index().to_dict()
decision_counts = explanations['decision_hint'].value_counts().sort_index().to_dict()
top_rows = explanations.head(8).to_dict(orient='records')
summary = {
    'schema_version': 'research.analysis_model_difference_explanations.v1',
    'status': 'completed',
    'source_card': str(SOURCE_CARD.relative_to(REPO_ROOT)),
    'baseline_model': baseline_model,
    'window_names': list(windows.keys()),
    'explanation_count': int(len(explanations)),
    'signal_counts': {str(key): int(value) for key, value in signal_counts.items()},
    'decision_counts': {str(key): int(value) for key, value in decision_counts.items()},
    'top_segments': top_rows,
    'csv_path': 'model_difference_explanations.csv',
    'notebook_review_note': '中文 review：该表用于解释不同 regime/window 下模型 winner/loser 和复核动作，不替代生产晋升验证。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.analysis_model_difference_explanations.v1',
 'status': 'completed',
 'source_card': 'research/quantbench_model_matrix_smoke_v1',
 'baseline_model': 'logistic_clf',
 'window_names': ['early_test', 'late_test', 'full_test'],
 'explanation_count': 19,
 'signal_counts': {'baseline_defends': 6,
  'segment_winner': 10,
  'small_gap': 3},
 'decision_counts': {'block_or_rework_loser': 6,
  'monitor': 3,
  'promote_segment_candidate': 10},
 'top_segments': [{'schema_version': 'analysis.model_difference_explanation.v1',
   'priority_rank': 1,
   'segment_type': 'window_regime',
   'segment': 'late_test::downtrend',
   'window': 'late_test',
   'regime': 'downtrend',
   'baseline_model': 'logistic_clf',
   'winner_model': 'alstm_mse',
   'loser_model': 'logistic_clf',
   'winner_relative_return_pct': 29.268236735322063,
   'loser_relative_return_pct': -18.630899214723712,
   'relative_return_pct_spread': 47.899135950045775,
   'winner_vs_baseline_delta': 47.8991359500

In [3]:
preview_columns = [
    'priority_rank',
    'segment_type',
    'segment',
    'baseline_model',
    'winner_model',
    'loser_model',
    'relative_return_pct_spread',
    'explanation_signal',
    'decision_hint',
    'review_note',
]
explanations.loc[:, preview_columns].head(12)

,priority_rank,segment_type,segment,baseline_model,winner_model,loser_model,relative_return_pct_spread,explanation_signal,decision_hint,review_note
0,1,window_regime,late_test::downtrend,logistic_clf,alstm_mse,logistic_clf,47.899136,segment_winner,promote_segment_candidate,中文 review：alstm_mse 在 late_test::downtrend 强于基...
1,2,regime,downtrend,logistic_clf,alstm_mse,mlp_mse,52.699142,segment_winner,promote_segment_candidate,中文 review：alstm_mse 在 downtrend 强于基准 logistic_...
2,3,window_regime,full_test::downtrend,logistic_clf,alstm_mse,mlp_mse,52.699142,segment_winner,promote_segment_candidate,中文 review：alstm_mse 在 full_test::downtrend 强于基...
3,4,all,all,logistic_clf,logistic_clf,mlp_mse,42.555385,baseline_defends,block_or_rework_loser,中文 review：基准 logistic_clf 在 all 领先，mlp_mse 是当前...
4,5,window,full_test,logistic_clf,logistic_clf,mlp_mse,42.555385,baseline_defends,block_or_rework_loser,中文 review：基准 logistic_clf 在 full_test 领先，mlp_m...
5,6,window,late_test,logistic_clf,extra_trees_rank,mlp_mse,34.496111,segment_winner,promote_segment_candidate,中文 review：extra_trees_rank 在 late_test 强于基准 lo...
6,7,window_regime,late_test::bull,logistic_clf,alstm_mse,mlp_mse,36.832691,segment_winner,promote_segment_candidate,中文 review：alstm_mse 在 late_test::bull 强于基准 log...
7,8,regime,stress,logistic_clf,extra_trees_rank,alstm_mse,33.871405,segment_winner,promote_segment_candidate,中文 review：extra_trees_rank 在 stress 强于基准 logis...
8,9,window_regime,full_test::stress,logistic_clf,extra_trees_rank,alstm_mse,33.871405,segment_winner,promote_segment_candidate,中文 review：extra_trees_rank 在 full_test::stress...
9,10,regime,bull,logistic_clf,catboost_rank,alstm_mse,24.656980,segment_winner,promote_segment_candidate,中文 review：catboost_rank 在 bull 强于基准 logistic_c...


In [4]:
conclusion = "## 中文结论\n\n- 做了什么：本 notebook 使用 `src/analysis.build_model_difference_explanations`，把 `logistic_clf` 作为基准，对 QuantBench smoke 回测的全区间、regime、固定窗口和 window-regime 分段生成模型差异解释表。\n- 核心指标：共生成 `19` 条解释；signal 分布为 `{'baseline_defends': 6, 'segment_winner': 10, 'small_gap': 3}`；decision 分布为 `{'block_or_rework_loser': 6, 'monitor': 3, 'promote_segment_candidate': 10}`。\n- regime 差异：最高优先级分段是 `late_test::downtrend`，winner=`alstm_mse`，loser=`logistic_clf`，相对收益 spread=`47.8991`。\n- 是否晋升：不晋升。本实验是 analysis 框架验证，只说明解释表可以作为 dashboard/notebook review 输入，不证明任何 smoke 模型可上线。\n- 风险：输入来自 smoke 回测，样本短且不代表完整 QuantBench 生产表现；解释表只做分段诊断，不替代成本、滑点、压力窗口和 production health gate。\n- 下一步：将 `model_difference_explanations.csv` 接入 dashboard view/filter 或 schedule 通知摘要，让 review 时直接看到每个 regime/window 的 winner、loser 和建议动作。"
(CARD / 'analysis_conclusion.md').write_text(conclusion + '\n', encoding='utf-8')
CARD / 'analysis_conclusion.md'

PosixPath('/Users/rziiiii/dev/workspace/research/analysis_model_difference_explanations_v1/analysis_conclusion.md')

## 中文结论

- 做了什么：本 notebook 使用 `src/analysis.build_model_difference_explanations`，把 `logistic_clf` 作为基准，对 QuantBench smoke 回测的全区间、regime、固定窗口和 window-regime 分段生成模型差异解释表。
- 核心指标：共生成 `19` 条解释；signal 分布为 `{'baseline_defends': 6, 'segment_winner': 10, 'small_gap': 3}`；decision 分布为 `{'block_or_rework_loser': 6, 'monitor': 3, 'promote_segment_candidate': 10}`。
- regime 差异：最高优先级分段是 `late_test::downtrend`，winner=`alstm_mse`，loser=`logistic_clf`，相对收益 spread=`47.8991`。
- 是否晋升：不晋升。本实验是 analysis 框架验证，只说明解释表可以作为 dashboard/notebook review 输入，不证明任何 smoke 模型可上线。
- 风险：输入来自 smoke 回测，样本短且不代表完整 QuantBench 生产表现；解释表只做分段诊断，不替代成本、滑点、压力窗口和 production health gate。
- 下一步：将 `model_difference_explanations.csv` 接入 dashboard view/filter 或 schedule 通知摘要，让 review 时直接看到每个 regime/window 的 winner、loser 和建议动作。
